Load datasets and preprocessor

In [4]:
import pandas as pd
import numpy as np

import sklearn
import joblib

sklearn.set_config(transform_output="pandas")
pd.set_option('display.max_rows', None)

results = {'library':[], 'algorithm':[], 'hyper_params':[],'AUC':[], 'Accuracy':[]}


transformer = joblib.load("transformer.pkl")
train_df    = pd.read_csv("data/balanced_train_data.csv")

y             = train_df['TARGET'].astype(np.float32)
train_df      = train_df.drop(columns=['TARGET'])
X_transformed = transformer.transform(train_df).astype(np.float32)


test_df = pd.read_csv("data/split_data_test.csv")
y_test = test_df['TARGET'].astype(np.float32)
X_test = test_df.drop(columns=['TARGET'])
X_test_transformed = transformer.transform(X_test).astype(np.float32)

del train_df, test_df, X_test


print(y.shape)
print(y_test.shape)
print(X_transformed.shape)
print(X_test_transformed.shape)

(227321,)
(71038,)
(227321, 53)
(71038, 53)


Naive Solution

In [5]:
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.dummy import DummyClassifier

naive_model = DummyClassifier(strategy="most_frequent")
naive_model.fit(X_transformed, y)


y_pred  = naive_model.predict(X_test_transformed)
y_probs = naive_model.predict_proba(X_test_transformed)[:, 1]

results['library'].append('sklearn')
results['algorithm'].append('DummyClassifier')
results['hyper_params'].append(None)
results['AUC'].append(roc_auc_score(y_test, y_probs))
results['Accuracy'].append(accuracy_score(y_test, y_pred))
print(results)


{'library': ['sklearn'], 'algorithm': ['DummyClassifier'], 'hyper_params': [None], 'AUC': [0.5], 'Accuracy': [0.918564711844365]}


Random Forest Solution

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
    
param_grid = {
    'n_estimators': [100],
    'max_depth': [None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1],
}

rclr = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)
cv = GridSearchCV(estimator=rclr, param_grid=param_grid, cv=3, n_jobs=-1, scoring='roc_auc_ovr',)

cv.fit(X=X_transformed, y=y)
y_probs = cv.predict_proba(X=X_test_transformed)[:, 1]
y_pred  = cv.predict(X=X_test_transformed)

results['library'].append('sklearn')
results['algorithm'].append('Random Forest Classifier + grid search')
results['hyper_params'].append(str(str(cv.best_params_)))
results['AUC'].append(roc_auc_score(y_test, y_probs))
results['Accuracy'].append(accuracy_score(y_test, y_pred))


In [ ]:
MLP Classifier Solution

In [8]:
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.neural_network import MLPClassifier

param_grid = {
    "activation": ["relu"],
    "alpha": [0.0001, 0.001],
    "learning_rate_init": [0.001],
    "solver": ["adam"]
}

model = MLPClassifier(
    early_stopping=True,
    solver='adam',
    activation='relu',
    max_iter=300,
    hidden_layer_sizes=(128, 64, 32),
    batch_size=32
)

cv = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='roc_auc_ovr',
    n_jobs=-1
)

sample_weight = compute_sample_weight(class_weight='balanced', y=y)
cv.fit(X_transformed, y, sample_weight=sample_weight)

y_probs = cv.predict_proba(X=X_test_transformed)[:, 1]
y_pred  = cv.predict(X=X_test_transformed)

results['library'].append('sklearn')
results['algorithm'].append('MLP Classifier + grid search')
results['hyper_params'].append(str(str(cv.best_params_)))
results['AUC'].append(roc_auc_score(y_test, y_probs))
results['Accuracy'].append(accuracy_score(y_test, y_pred))


In [ ]:
Load validation dataset

In [10]:
val_df = pd.read_csv("data/split_data_val.csv")

y_val  = val_df['TARGET'].copy().astype(np.float32)
val_df = val_df.drop(columns=['TARGET'])
X_val  = transformer.transform(val_df).astype(np.float32)

del val_df

print(y_val.shape)
print(X_val.shape)

(56831,)
(56831, 53)


Keras solution

In [18]:
import tensorflow as tf
import keras_tuner as kt

from keras.losses import BinaryCrossentropy
from keras.layers import Dense, Dropout, Input
from keras.models import Sequential
from keras.optimizers import Adam
from keras.models import Sequential
from keras.metrics import AUC
from keras.callbacks import EarlyStopping

from sklearn.utils import class_weight

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
tf.get_logger().setLevel('ERROR')


weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

dict_weights = dict(enumerate(weights))

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=20, 
    restore_best_weights=True
)

def build_model(hp):
    model = Sequential([
        Input(shape=(X_transformed.shape[1],)),
        Dense(
            units=hp.Int('units', min_value=64, max_value=128, step=64),
            activation="relu",
        ),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dropout(0.1),
        Dense(16, activation="relu"),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[AUC(name='auc'), 'accuracy']
    )
    return model


tuner = kt.GridSearch(
    build_model,
    objective=kt.Objective("val_auc", direction="max"),
    directory="tuner_dir",
    project_name="auc_search"
)

tuner.search(
    X_transformed, 
    y, 
    batch_size=64, 
    epochs=200, 
    validation_split=0.2,
    class_weight=dict_weights,
    callbacks=[early_stop],
    validation_data=(X_val, y_val)
)

model    = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

y_prob = model.predict(X_test_transformed).ravel()
y_pred = (y_prob > 0.5).astype(int)

results['library'].append('Keras')
results['algorithm'].append('multilayer-perceptrone + grid search')
results['hyper_params'].append(str(best_hps.values))
results['AUC'].append(roc_auc_score(y_test, y_probs))
results['Accuracy'].append(accuracy_score(y_test, y_pred))
model.summary()

Reloading Tuner from tuner_dir\auc_search\tuner0.json
2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 524us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           3,456 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,081 (23.75 KB)

 Trainable params: 6,081 (23.75 KB)

 Non-trainable params: 0 (0.00 B)

TensorFlow Solution

In [19]:
from lib.tnn_wrapper import TNetWrapper

import warnings
warnings.filterwarnings("ignore")

param_grid = {
    "hidden_units": [32],
    "epochs": [10]
}

grid = GridSearchCV(
    TNetWrapper(),
    param_grid=param_grid,
    scoring='roc_auc_ovr',
    n_jobs=-1
)

grid.fit(X_transformed, y)

best_model = grid.best_estimator_

y_prob = best_model.predict_proba(X_test_transformed).numpy()
y_pred = (y_prob > 0.5).astype(int)

best_params = {
    "hidden_units":best_model.hidden_units,
    "epochs" : best_model.epochs,
    "learning_rate" : best_model.learning_rate,
    "batch_size" : best_model.batch_size
}

results['library'].append('TensorFlow')
results['algorithm'].append('multilayer-perceptrone + grid search')
results['hyper_params'].append(str(best_params))
results['AUC'].append(roc_auc_score(y_test, y_probs))
results['Accuracy'].append(accuracy_score(y_test, y_pred))


Epoch 0, Average Loss: 0.3378
Epoch 1, Average Loss: 0.2831
Epoch 2, Average Loss: 0.2732
Epoch 3, Average Loss: 0.2674
Epoch 4, Average Loss: 0.2635
Epoch 5, Average Loss: 0.2605
Epoch 6, Average Loss: 0.2581
Epoch 7, Average Loss: 0.2560
Epoch 8, Average Loss: 0.2542
Epoch 9, Average Loss: 0.2527


Numpy Solution

In [21]:
from lib.nn_wrapper import NNetWrapper

param_grid = {
    "drop_rate": [0.1, 0.2],
    "early_stopping_patience" : [40]
}

grid = GridSearchCV(
    estimator=NNetWrapper(),
    param_grid=param_grid,
    scoring='roc_auc_ovr',
    n_jobs=-1,
)

grid.fit(X_transformed, y, validation_data=(X_val, y_val))


best_model = grid.best_estimator_

y_prob    = best_model.predict_proba(X_test_transformed)
y_pred    = (y_prob > 0.5).astype(int)

best_params  = {
        'learning_rate':best_model.learning_rate,
        'epochs':best_model.epochs,
        'batch_size':best_model.batch_size,
        'hidden_layers':'(64, 32, 16)',
        'early_stopping': best_model.early_stopping_patience,
        'drop_rate' : best_model.drop_rate,
        'restore_best_weights': best_model.restore_best_weights
}
results['library'].append('numpy')
results['algorithm'].append('multilayer-perceptrone + grid search')
results['hyper_params'].append(str(best_params))
results['AUC'].append(roc_auc_score(y_test, y_probs))
results['Accuracy'].append(accuracy_score(y_test, y_pred))


Epoch 1/300
val loss : 12.820973828168569 Validation loss decreased. Resetting counter.


100%|===============================================================================================



Epoch 2/300
val loss : 0.289944829351158 Validation loss decreased. Resetting counter.


100%|===============================================================================================



Epoch 3/300
val loss : 0.25109897212357657 Validation loss decreased. Resetting counter.


...


Epoch 81/300
val loss : 0.2253875599846608 EarlyStopping counter: 38 out of 40


100%|===============================================================================================



Epoch 82/300
val loss : 0.22516305581526397 EarlyStopping counter: 39 out of 40


100%|===============================================================================================



Epoch 83/300
val loss : 0.22538267662843578 EarlyStopping counter: 40 out of 40


Results table

In [23]:
table = pd.DataFrame(data=results)
print(table)

      library                               algorithm  \
0     sklearn                         DummyClassifier   
1     sklearn  Random Forest Classifier + grid search   
2     sklearn            MLP Classifier + grid search   
3       Keras    multilayer-perceptrone + grid search   
4  TensorFlow    multilayer-perceptrone + grid search   
5       numpy    multilayer-perceptrone + grid search   

                                        hyper_params       AUC  Accuracy  
0                                               None  0.500000  0.918565  
1  {'max_depth': None, 'min_samples_leaf': 1, 'mi...  0.827272  0.918621  
2  {'activation': 'relu', 'alpha': 0.001, 'learni...  0.832158  0.703623  
3                                      {'units': 64}  0.832158  0.703623  
4  {'hidden_units': 32, 'epochs': 10, 'learning_r...  0.832158  0.918283  
5  {'learning_rate': 0.001, 'epochs': 300, 'batch...  0.832158  0.918903  
